In [1]:
# ============================================================
# ARC-CHALLENGE — Full generation pipeline, self-contained
# Incorporates every lesson from tonight's GSM8K run:
#   - Higher token budget (768) to avoid truncation from the start
#   - Careful answer extraction with multiple fallback tiers
#   - Resumable, per-example saving
#   - Fixed random seed for reproducibility
# Needs GPU. Run in a fresh session or your Generation notebook.
# ============================================================

In [2]:
# --- CELL 1: setup ---
import os, json, re, hashlib, random

BASE_DIR = "/kaggle/working/spectral_v2"
DIRS = {
    "generations": f"{BASE_DIR}/generations", "extractions": f"{BASE_DIR}/extractions",
    "features": f"{BASE_DIR}/features", "results": f"{BASE_DIR}/results", "logs": f"{BASE_DIR}/logs",
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

ARC_DIR = f"{DIRS['generations']}/arc_challenge"
os.makedirs(ARC_DIR, exist_ok=True)

In [3]:
# --- CELL 2: GPU + model load ---
import torch
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "STOP: turn GPU on."

from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, attn_implementation="eager", torch_dtype=torch.float16, device_map="auto",
)
model.eval()
torch.manual_seed(42)  # fixes reproducibility issue found earlier tonight
assert model.config._attn_implementation == "eager"
print("Model loaded.")

CUDA available: True


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded.


In [4]:
# --- CELL 3: generate_and_replay function (unchanged from GSM8K) ---
def generate_and_replay(prompt_text, model, tokenizer, max_new_tokens=768, temperature=0.3):
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        gen_ids = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=True,
            temperature=temperature, pad_token_id=tokenizer.eos_token_id,
        )
    full_input_ids = gen_ids
    total_len = full_input_ids.shape[1]
    response_len = total_len - prompt_len
    with torch.no_grad():
        replay_out = model(input_ids=full_input_ids, output_attentions=True, output_hidden_states=True)
    return {
        "full_input_ids": full_input_ids, "attentions": replay_out.attentions,
        "hidden_states": replay_out.hidden_states, "prompt_len": prompt_len,
        "response_len": response_len, "total_len": total_len,
    }

In [5]:
# --- CELL 4: ARC-specific answer extraction (letter, not number) ---
def extract_arc_answer(generated_text):
    """
    ARC answers are single letters (A/B/C/D, sometimes 1/2/3/4).
    Priority 1: \boxed{X}
    Priority 2: bold **X**
    Priority 3: "answer is X" / "answer: X" pattern
    Priority 4: last standalone capital letter A-D in the text
    """
    boxed_matches = re.findall(r"\\boxed\{([A-Da-d1-4])\}", generated_text)
    if boxed_matches:
        return boxed_matches[-1].upper(), "boxed"

    bold_matches = re.findall(r"\*\*([A-Da-d1-4])\*\*", generated_text)
    if bold_matches:
        return bold_matches[-1].upper(), "bold"

    answer_is_matches = re.findall(r"[Aa]nswer\s*(?:is|:)\s*\(?([A-Da-d1-4])\)?", generated_text)
    if answer_is_matches:
        return answer_is_matches[-1].upper(), "answer_is_pattern"

    stripped = re.sub(r"\\\[.*?\\\]", " ", generated_text, flags=re.DOTALL)
    letter_matches = re.findall(r"\b([A-D])\b", stripped)
    if letter_matches:
        return letter_matches[-1], "fallback_last_letter"

    return None, "none_found"

In [6]:
# --- CELL 5: load ARC-Challenge, sample 150 ---
from datasets import load_dataset

arc = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="test")
print(f"ARC-Challenge test set size: {len(arc)}")
print("Example:", arc[0])

N_EXAMPLES = 150
random.seed(42)
indices = random.sample(range(len(arc)), N_EXAMPLES)
subset = arc.select(indices)
print(f"Selected {N_EXAMPLES} examples (seed=42).")

# --- CELL 6: resumable generation loop ---
def example_id(question_text):
    return hashlib.sha256(question_text.encode()).hexdigest()[:12]

def already_done(ex_id):
    return os.path.exists(f"{ARC_DIR}/{ex_id}.json")

def save_example(ex_id, data):
    tmp_path = f"{ARC_DIR}/{ex_id}.json.tmp"
    final_path = f"{ARC_DIR}/{ex_id}.json"
    with open(tmp_path, "w") as f:
        json.dump(data, f)
    os.rename(tmp_path, final_path)

import numpy as np

completed = 0
skipped = 0
failed = 0
truncated_count = 0

for i, ex in enumerate(subset):
    question = ex["question"]
    choices_text = ex["choices"]["text"]
    choices_label = ex["choices"]["label"]
    ground_truth = ex["answerKey"]  # e.g. "A", "B", "1", "2"

    ex_id = example_id(question)
    if already_done(ex_id):
        skipped += 1
        continue

    try:
        choices_formatted = "\n".join(f"{label}. {text}" for label, text in zip(choices_label, choices_text))
        full_question = f"{question}\n\n{choices_formatted}\n\nThink step by step, then give your final answer as a single letter."

        prompt = tokenizer.apply_chat_template(
            [{"role": "user", "content": full_question}],
            tokenize=False, add_generation_prompt=True,
        )

        gen_result = generate_and_replay(prompt, model, tokenizer, max_new_tokens=768, temperature=0.3)

        response_text = tokenizer.decode(
            gen_result["full_input_ids"][0][gen_result["prompt_len"]:], skip_special_tokens=True
        )
        model_answer, method = extract_arc_answer(response_text)
        is_correct = (model_answer == ground_truth) if model_answer is not None else False

        is_truncated = gen_result["response_len"] >= 766
        if is_truncated:
            truncated_count += 1

        tensor_path = f"{ARC_DIR}/{ex_id}_tensors.npz"
        attn_stack = np.stack([a[0].cpu().numpy() for a in gen_result["attentions"]])
        hs_stack = np.stack([h[0].cpu().numpy() for h in gen_result["hidden_states"]])
        np.savez_compressed(tensor_path, attentions=attn_stack.astype(np.float16), hidden_states=hs_stack.astype(np.float16))

        record = {
            "example_id": ex_id, "dataset": "arc_challenge",
            "question": question, "choices_text": choices_text, "choices_label": choices_label,
            "ground_truth": ground_truth, "response_text": response_text,
            "model_answer": model_answer, "extraction_method": method, "is_correct": is_correct,
            "prompt_len": gen_result["prompt_len"], "response_len": gen_result["response_len"],
            "total_len": gen_result["total_len"], "is_truncated": is_truncated,
            "tensor_path": tensor_path,
        }
        save_example(ex_id, record)
        completed += 1

        if (i + 1) % 10 == 0:
            print(f"[{i+1}/{N_EXAMPLES}] completed={completed} skipped={skipped} failed={failed} truncated={truncated_count}")

    except Exception as e:
        failed += 1
        print(f"FAILED on example {i} ({ex_id}): {e}")
        continue

print(f"\nDone. completed={completed} skipped={skipped} failed={failed} truncated={truncated_count}")
print(f"Total examples on disk: {len([f for f in os.listdir(ARC_DIR) if f.endswith('.json')])}")

print("\n>>> SAVE & RUN ALL NOW -- this cost real GPU time. <<<")

README.md: 0.00B [00:00, ?B/s]

ARC-Challenge/train-00000-of-00001.parqu(…):   0%|          | 0.00/190k [00:00<?, ?B/s]

ARC-Challenge/test-00000-of-00001.parque(…):   0%|          | 0.00/204k [00:00<?, ?B/s]

ARC-Challenge/validation-00000-of-00001.(…):   0%|          | 0.00/55.7k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1119 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1172 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/299 [00:00<?, ? examples/s]

ARC-Challenge test set size: 1172
Example: {'id': 'Mercury_7175875', 'question': 'An astronomer observes that a planet rotates faster after a meteorite impact. Which is the most likely effect of this increase in rotation?', 'choices': {'text': ['Planetary density will decrease.', 'Planetary years will become longer.', 'Planetary days will become shorter.', 'Planetary gravity will become stronger.'], 'label': ['A', 'B', 'C', 'D']}, 'answerKey': 'C'}
Selected 150 examples (seed=42).
[10/150] completed=10 skipped=0 failed=0 truncated=0
[20/150] completed=20 skipped=0 failed=0 truncated=0
[30/150] completed=30 skipped=0 failed=0 truncated=0
[40/150] completed=40 skipped=0 failed=0 truncated=0
[50/150] completed=50 skipped=0 failed=0 truncated=0
[60/150] completed=60 skipped=0 failed=0 truncated=0
[70/150] completed=70 skipped=0 failed=0 truncated=0
[80/150] completed=80 skipped=0 failed=0 truncated=0
[90/150] completed=90 skipped=0 failed=0 truncated=0
[100/150] completed=100 skipped=0 fai